# 02 · 핵심 분석: 교차표·집단비교·회귀  (모듈 3)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amnotyoung/oda-ai-stats/blob/main/notebooks/02_core_analysis.ipynb)

> 대표 분석 3종을 Python으로 수행하고, **같은 분석을 STATA로도**(각 절 🔁) 해서
> 두 결과의 숫자가 같은지 **교차검증**한다. — "숫자는 같다. 출력 모양만 다르다."

In [ ]:
# [한 번 실행하고 넘어가세요] 한글 폰트 설정 — Colab 그래프의 한글이 깨지지 않도록(처음 1회 ~20초)
import os, matplotlib.pyplot as plt, matplotlib.font_manager as fm
try:
    p = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"   # 나눔고딕 폰트 경로
    if not os.path.exists(p):                                # 없으면
        os.system("apt-get -qq install -y fonts-nanum > /dev/null 2>&1")  # 설치
    fm.fontManager.addfont(p)                               # matplotlib에 폰트 등록
    plt.rc("font", family="NanumGothic")                   # 기본 글꼴로 지정
except Exception:
    pass                                                    # 실패해도 그냥 진행(영문은 정상)
plt.rc("axes", unicode_minus=False)                         # 마이너스 기호 깨짐 방지

In [ ]:
import pandas as pd, numpy as np
from scipy import stats                       # t검정·ANOVA 등 통계 검정 함수
import statsmodels.formula.api as smf         # 회귀분석(R 스타일 수식 사용)
df = pd.read_csv("https://raw.githubusercontent.com/amnotyoung/oda-ai-stats/main/data/wdi_panel.csv")                      # 데이터 불러오기
df["log_gdp"] = np.log(df["gdp_pc"])           # 로그변환 변수 미리 생성
df["log_pop"] = np.log(df["pop"])
print("준비 완료:", df.shape)

## 1) 교차표 — 지역 × 소득그룹 (국가 수)
지역·소득은 국가 속성이므로 **국가 단위로** 집계한다(연도 중복 제거).

In [ ]:
# 지역·소득은 '국가'의 속성 → 국가당 1행만 남겨 연도 중복 제거
countries = df.drop_duplicates("economy")
# 행=지역, 열=소득그룹 으로 국가 수를 센 교차표
pd.crosstab(countries["region_name"], countries["income_name"])

> 🖼️ **그림으로**: 위 교차표의 숫자를 **색의 진하기**로 — 어느 지역에 어떤 소득국이 몰렸는지 한눈에.

In [ ]:
# 교차표를 히트맵으로 — 숫자 표가 '패턴'으로 보인다(진할수록 국가 많음)
import matplotlib.pyplot as plt
ct = pd.crosstab(countries["region_name"], countries["income_name"])   # 위 교차표 다시 계산
fig, ax = plt.subplots(figsize=(8.5, 4.5))
im = ax.imshow(ct.values, cmap="Blues", aspect="auto")                 # 값을 색으로(파랑 진할수록 큼)
ax.set_xticks(range(ct.shape[1])); ax.set_xticklabels(ct.columns, rotation=20, ha="right")  # 열 = 소득그룹
ax.set_yticks(range(ct.shape[0])); ax.set_yticklabels(ct.index)                              # 행 = 지역
for i in range(ct.shape[0]):                                           # 각 칸에 실제 국가 수 숫자 표기
    for j in range(ct.shape[1]):
        ax.text(j, i, ct.values[i, j], ha="center", va="center", fontsize=9)
ax.set_title("지역 × 소득그룹 국가 수 (진할수록 많음)")
plt.colorbar(im, ax=ax, shrink=0.8); plt.tight_layout(); plt.show()

🔁 **STATA**: `tabulate region_name income_name`  ·  코드 → `stata/02_crosstab.do`

## 2) 집단 비교 — 지역 격차(t검정) · 소득그룹 차이(ANOVA)
- **t검정** = *두* 집단의 평균이 다른지, **ANOVA** = *셋 이상* 집단의 평균이 다른지 검정.
- 둘 다 **`p` < 0.05면 "우연으로 보기 어렵다(통계적으로 유의)"**.

In [ ]:
# t검정: 두 집단(사하라이남 아프리카 vs 그 외)의 기대수명 평균 차이
ssa  = df.loc[df.region_name == "Sub-Saharan Africa", "life_exp"].dropna()   # 사하라이남 값만
rest = df.loc[df.region_name != "Sub-Saharan Africa", "life_exp"].dropna()   # 나머지 값만
t, p = stats.ttest_ind(ssa, rest, equal_var=False)   # 분산이 달라 Welch 검정(STATA는 ttest ..., unequal)
print(f"[t검정] 사하라이남={ssa.mean():.1f}세  그외={rest.mean():.1f}세  t={t:.1f}  p={p:.1e}")  # p 작으면 차이 유의

# ANOVA: 셋 이상 집단(소득그룹 4개) 간 기대수명 평균 차이
groups = [g["life_exp"].dropna().values for _, g in df.groupby("income_name")]  # 소득그룹별 값 묶음
F, p2 = stats.f_oneway(*[g for g in groups if len(g) > 1])   # 일원분산분석(F가 클수록 차이 큼)
print(f"[ANOVA] 소득그룹별 기대수명  F={F:.0f}  p={p2:.1e}")

> 🖼️ **그림으로**: 검정이 본 '평균 차이'를 **분포(상자)**로 — 상자가 겹치는지 떨어지는지 눈으로 확인.

In [ ]:
# 소득그룹별 기대수명 분포 — ANOVA의 'p<0.05(그룹 간 차이 유의)'를 그림으로 확인
order  = ["Low income", "Lower middle income", "Upper middle income", "High income"]
labels = ["저소득", "중하위", "중상위", "고소득"]
data   = [df.loc[df["income_name"] == g, "life_exp"].dropna() for g in order]   # 그룹별 값 묶음
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.boxplot(data, showmeans=True)                                 # 가운데선 = 중앙값, 삼각 = 평균
ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels(labels)
ax.set_ylabel("기대수명 (세)"); ax.set_title("소득그룹별 기대수명 분포 — 상자가 위로 갈수록 길어진다")
plt.tight_layout(); plt.show()

🔁 **STATA**: `ttest life_exp, by(ssa) unequal`  ·  `oneway life_exp income_n`  ·  코드 → `stata/03_group_compare.do`

## 3) 회귀분석 — Preston 곡선 (소득 → 기대수명)
**회귀** = 한 변수(기대수명)를 다른 변수(소득)로 설명하는 **직선**을 찾는 것. 그런데 소득↔기대수명은
휘어 있어(Preston 곡선: 저소득 급증·고소득 평평) 직선이 아니다 → **log(소득)**으로 펴서 선형화한다
(OLS는 변수 정규성을 요구하지 않는다; 왜도·이상치 완화는 덤) — AI가 자동으로 안 할 수 있는 **설계 판단**.

> **출력 읽는 법**: `coef`=기울기(log소득 1↑당 수명 변화), `P>|t|`<0.05면 유의, `R²`=설명력(0~1).

In [ ]:
# "life_exp ~ log_gdp" = 기대수명을 log소득으로 설명. ols=최소제곱법, .fit()=계수 추정
m = smf.ols("life_exp ~ log_gdp", data=df).fit()
print(m.summary().tables[1])      # 계수표: coef(기울기)·std err(오차)·P>|t|(유의확률)
print(f"\nR² = {m.rsquared:.3f}")  # 설명력(0~1): 소득이 기대수명 변동의 몇 %를 설명하나

> 🖼️ **그림으로**: 회귀선을 데이터 위에 직접 그린다 — *왜 로그변환이 필요한지*가 한눈에 보인다(Preston 곡선).

In [ ]:
# Preston 곡선 — 회귀를 '직선 그리기'로 직접 본다. 로그변환 전후를 나란히 비교
xs = np.linspace(df["log_gdp"].min(), df["log_gdp"].max(), 100)   # 회귀선을 그릴 x 좌표들
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
# (왼쪽) 변환 전 — 원본 소득축: 저소득서 급증·고소득서 평평한 '휜 곡선'
ax[0].scatter(df["gdp_pc"], df["life_exp"], s=6, alpha=0.4)
ax[0].set_xlabel("1인당 GDP (원본)"); ax[0].set_ylabel("기대수명")
ax[0].set_title("변환 전 — 휜 관계(Preston 곡선)")
# (오른쪽) 변환 후 — log축: 거의 직선 → 빨간 회귀선이 잘 맞는다
ax[1].scatter(df["log_gdp"], df["life_exp"], s=6, alpha=0.4, c="green")
ax[1].plot(xs, m.params["Intercept"] + m.params["log_gdp"] * xs, "r-", lw=2)  # 추정된 회귀직선
ax[1].set_xlabel("log(1인당 GDP)"); ax[1].set_ylabel("기대수명")
ax[1].set_title("변환 후 — 직선화 + 회귀선")
plt.tight_layout(); plt.show()

**해석**: `log_gdp` 계수가 **양수** → 소득이 높을수록 기대수명↑. (소득이 e배면 ≈ +계수 만큼 수명)
R²가 높아 소득만으로 기대수명 차이의 상당 부분이 설명된다.

🔁 **STATA**: `regress life_exp log_gdp` — **한 줄**.  코드 → `stata/04_regression.do`
> 양쪽 계수·R²가 같은지 **교차검증**: 같으면 신뢰, 다르면 하나가 틀린 것.

---
✅ **핵심**: 교차표·검정·회귀를 입문자가 첫날 직접 돌렸다. 비결은 *문법 암기*가 아니라
*무엇을·왜 분석할지 설계*하고 *검증·해석*하는 것.